# 05 · Checks and audits

Independent checks behind the statements in the reports, from `scripts/fink_checks.py`:

1. **Raw re-derivation.** Template magnitudes, panel samples and Fink labels recomputed from the raw alerts with
   pandas, with no catalog views or SQL, and compared with notebook 01.
2. **Crossmatch failures.** How often Fink's SIMBAD crossmatch returned a type, nothing within 1″, or `"Fail"` (a CDS
   service error).
3. **RR Lyrae and labels.** Where known RR Lyrae sit in u−g, what Fink's "Periodic" label really contains, u-band
   reference populations, and AGN shares per sample.
4. **u-band conversion.** RR Lyrae u−g converted to the SDSS system with Rubin RTN-099 (LSSTComCam relations).
5. **Magnitude assembly.** Alerts per band, same-night coverage and template-flux spread.

**Inputs:** notebooks 01 and 02, and the catalog's pinned raw files.
**Outputs:** `RUN_ROOT/05_checks_and_audits/tables/*.json`. Run time: about 5 minutes.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import fink_checks
template_photometry = require(RUN_ROOT / "01_template_colours/tables/dia_object_template_photometry.parquet", "01_template_colours")
science_table = require(RUN_ROOT / "02_science_suite/tables/dia_object_science_table.parquet", "02_science_suite")
out = step_dir("05_checks_and_audits")
raw_files = fink_checks.pinned_raw_files(CATALOG, DATA_ROOT)
print(len(raw_files), "pinned raw Parquet files")

## 1 · Raw re-derivation
Expected: no band-presence, panel or label mismatches. Magnitude differences of about 6.6e-5 come only from using the
exact AB zero point (31.40007) here instead of the rounded 31.4.

In [ ]:
audit = fink_checks.raw_template_rederivation(raw_files, template_photometry)
save_json(audit, out / "tables" / "raw_rederivation.json")
audit

## 2 · SIMBAD crossmatch: match, no match, or failed

In [ ]:
xm = fink_checks.crossmatch_failure_stats(raw_files)
save_json(xm, out / "tables" / "crossmatch_failures.json")
xm

## 3 · RR Lyrae, the "Periodic" label, u-band references and AGN shares

In [ ]:
labels = fink_checks.rr_lyrae_and_label_checks(template_photometry, science_table)
save_json(labels, out / "tables" / "rr_lyrae_labels_agn.json")
labels

## 4 · RR Lyrae u−g in the SDSS system (Rubin RTN-099)

In [ ]:
rtn = fink_checks.rtn099_u_conversion(template_photometry, science_table)
save_json(rtn, out / "tables" / "rtn099_u_conversion.json")
{k: v for k, v in rtn.items() if k != "per_object"}

## 5 · How multi-band magnitudes are assembled from single-band alerts

In [ ]:
facts = fink_checks.magnitude_assembly_facts(CATALOG, DATA_ROOT, template_photometry)
save_json(facts, out / "tables" / "magnitude_assembly.json")
facts